In [1]:
from nichenetpy.utils import (
    read_csv_cols,
)
from nichenetpy.parameter_optimization import construct_and_evaluate

import os
import requests
import pandas as pd
import session_info
import json
import numpy as np

In [2]:
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in (
    "gr_human.csv",
    "lr_network_human.csv",
    "lr_sig_human.csv",
    "optimized_source_weights.csv",
    "annotation_data_sources.csv"
):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [3]:
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

In [4]:
train_path = "D:/Data/nichenetpy/model_optimization"
with open(os.path.join(train_path, "settings_training_f1234.json"), "rb") as file:
    settings_CV = json.loads(file.read())
settings = settings_CV["settings"]

In [5]:
gr_network = gr_network[
    ((gr_network["database"] == "NicheNet_LT") & np.array([fr not in settings_CV["forbidden_ligands_nichenet"] for fr in gr_network["from"]]))
    |
    ((gr_network["database"] == "CytoSig") & np.array([fr not in settings_CV["forbidden_ligands_cytosig"] for fr in gr_network["from"]]))
]

In [6]:
eval = construct_and_evaluate(
    lr_network,
    gr_network,
    sig_network,
    settings
)

c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\sklearn\metrics\_ranking.py:1030: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\sklearn\metrics\_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\metrics.py:160: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pcc = pearsonr(response, prediction).statistic
c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\sklearn\metrics\_ranking.py:1030: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\sklearn\metrics\_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value

In [13]:
eval["performances_target_prediction"]["ifna_ifng_timeseries"]

{'auroc': np.float64(0.5309944398089709),
 'pearson': np.float64(0.021982905484978048),
 'aupr': np.float64(0.012008662910100306),
 'aupr_corrected': np.float64(0.0041590827713635405)}

In [10]:
eval["performances_ligand_prediction_single"][0]

,aupr,aupr_corrected,auroc,pearson,metric,group,ligand
0,0.016129,0.000256,0.516129,-0.083820,aupr,GSE6085_IL2_timeseries,IL2
1,0.016129,0.000256,0.516129,-0.083820,aupr_corrected,GSE6085_IL2_timeseries,IL2
2,0.012500,-0.003373,0.370968,-0.071186,auroc,GSE6085_IL2_timeseries,IL2
3,0.013514,-0.002360,0.419355,-0.069149,pearson,GSE6085_IL2_timeseries,IL2


In [9]:
session_info.show()